In [10]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework6") \
    .getOrCreate()

# Path data - sesuaikan jika perlu
DATA_PATH = os.path.expanduser("~/learn/homework6/yellow_tripdata_2025-11.parquet")
ZONE_PATH = os.path.expanduser("~/learn/homework6/taxi_zone_lookup.csv")
OUTPUT_PATH = os.path.expanduser("~/learn/homework6/yellow_nov_2025")

print("Spark version:", spark.version)

✅ Spark version: 3.3.2


In [11]:
print("=" * 40)
print("Q1 - Spark Version:", spark.version)
print("=" * 40)

Q1 - Spark Version: 3.3.2


In [12]:
df = spark.read.parquet(DATA_PATH)
print("Total rows:", df.count())

df.repartition(4).write.parquet(OUTPUT_PATH, mode="overwrite")

files = [f for f in os.listdir(OUTPUT_PATH) if f.endswith(".parquet")]
sizes = [os.path.getsize(f"{OUTPUT_PATH}/{f}") / (1024*1024) for f in files]

for f, s in zip(files, sizes):
    print(f"  {f}: {s:.2f} MB")

avg = sum(sizes) / len(sizes)
print(f"\n✅ Q2 - Average size: {avg:.2f} MB")

Total rows: 4181444


[Stage 13:>                                                         (0 + 4) / 4]

  part-00001-32ff0583-2589-4e0e-b065-f099d9f14281-c000.snappy.parquet: 26.35 MB
  part-00000-32ff0583-2589-4e0e-b065-f099d9f14281-c000.snappy.parquet: 26.36 MB
  part-00003-32ff0583-2589-4e0e-b065-f099d9f14281-c000.snappy.parquet: 26.37 MB
  part-00002-32ff0583-2589-4e0e-b065-f099d9f14281-c000.snappy.parquet: 26.36 MB

✅ Q2 - Average size: 26.36 MB


In [21]:
from pyspark.sql import functions as F

result = df.filter(
    F.to_date(F.col("tpep_pickup_datetime")) == "2025-11-15"
).count()

print(f"Q3 - Trips on Nov 15: {result:,}")

[Stage 26:>                                                         (0 + 4) / 4]

Q3 - Trips on Nov 15: 160,996


In [22]:
result = df.select(
    F.round(
        F.max(
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
        ), 2
    ).alias("max_hours")
)

print("Q4 - Longest trip:")
result.show()

Q4 - Longest trip:
+---------+
|max_hours|
+---------+
|    90.65|
+---------+



In [23]:
print("Q5 - Spark UI Port: 4040")
print("Buka: http://localhost:4040")

Q5 - Spark UI Port: 4040
Buka: http://localhost:4040


In [24]:
zones = spark.read.option("header", "true").csv(ZONE_PATH)

result = df.groupBy("PULocationID").count() \
    .join(
        zones.withColumnRenamed("LocationID", "PULocationID"),
        on="PULocationID"
    ) \
    .orderBy("count") \
    .select("Zone", "count") \
    .limit(5)

print("Q6 - Least frequent pickup zones:")
result.show(truncate=False)

Q6 - Least frequent pickup zones:
+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Green-Wood Cemetery                          |4    |
+---------------------------------------------+-----+



In [ ]:
spark.stop()
print("Spark session stopped.")